# Baseline ResNet-50 with 5-Fold Cross-Validation in Google Colab

This notebook trains a **baseline ResNet-50 without segmentation guidance** using parenchyma images from `000_dataset_v2`. The data is read from a `.tar.gz` archive in Google Drive, copied to Colab local storage, and validated before training starts. The training results and a copy of the JSON configuration are saved back to Google Drive.

Before starting, push the latest code to the configured branch, then select **Runtime > Change runtime type > GPU**. Run all cells in order from top to bottom. The `tqdm` progress bars are visible while copying the archive, validating the data, and running training and validation.

> Note: each `EXPERIMENT_ID` can only be used for one new training run. Change its value to create another experiment.

## Prepare the archive before opening Google Colab

Run the following command from the repository root on your local computer. The archive only contains the cross-validation metadata, parenchyma images, and ground-truth masks. **Probability maps are not included and are not used by the baseline ResNet-50.** The masks are only required for the optional XAI stage; they are not model inputs during training.

```bash
tar -czf a0d90f9e-3dd4-4de0-98af-12858696f613_cv_resnet50_parenchyma.tar.gz \
  000_dataset_v2/_segmentation_dataset/004_classification_cv_5fold_seed42.csv \
  000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma \
  000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma \
  000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask \
  000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask
```

When it is ready, upload the file to:

`MyDrive/mask-guided-lung-nodule-xai/a0d90f9e-3dd4-4de0-98af-12858696f613_cv_resnet50_parenchyma.tar.gz`

The directory structure inside the archive must start with `000_dataset_v2`. Do not include a probability-map directory.

## 1. Check the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. Enable a GPU in the Colab runtime settings."
    )

print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive

The dataset archive is read from Google Drive. All experiment results are also saved there so they are not lost when the Colab runtime stops.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Configure the experiment

This is the main configuration cell. Change the repository, archive location, experiment name, model, training, optimizer, or DataLoader settings here before running the next cell. `CT_PATH_COLUMN` intentionally uses `ct_parenchyma_path`.

In [ ]:
from pathlib import Path

# Repository
REPOSITORY_URL = (
    "https://github.com/FillipusAditya/"
    "mask-guided-lung-nodule-xai.git"
)
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Experiment
EXPERIMENT_ID = "a0d90f9e-3dd4-4de0-98af-12858696f613"
EXPERIMENT_COMPONENT = "classification/cv_resnet50"

# Dataset
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai"
)
ARCHIVE_NAME = f"{EXPERIMENT_ID}_cv_resnet50_parenchyma.tar.gz"
DRIVE_ARCHIVE_PATH = DRIVE_PROJECT_ROOT / ARCHIVE_NAME
LOCAL_ARCHIVE_PATH = Path("/content") / ARCHIVE_NAME
EXTRACTION_ROOT = Path("/content/classification_training_data")
DATASET_ROOT = EXTRACTION_ROOT / "000_dataset_v2/_segmentation_dataset"
COPY_ARCHIVE_TO_LOCAL = True
FORCE_EXTRACT = False

# Output
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "experiment_results"
DRIVE_OUTPUT_DIR = (
    DRIVE_OUTPUT_ROOT / EXPERIMENT_ID / EXPERIMENT_COMPONENT
)

# Input data
METADATA_FILENAME = "004_classification_cv_5fold_seed42.csv"
CT_PATH_COLUMN = "ct_parenchyma_path"
INPUT_HEIGHT = 224
INPUT_WIDTH = 224
NUM_FOLDS = 5
CLASS_TO_IDX = {"benign": 0, "malignant": 1}
NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
NORMALIZATION_STD = [0.229, 0.224, 0.225]

# Model ResNet-50
PRETRAINED_WEIGHTS = "DEFAULT"
CLASSIFIER_DROPOUT = 0.3

# Training
NUM_EPOCHS = 100
BATCH_SIZE = 32  # Reduce this value if the GPU runs out of memory.
LEARNING_RATE = 1e-3
SEED = 42
TRANSFORM_SEED = 42
CLASSIFICATION_THRESHOLD = 0.5
DEVICE = "auto"

# Optimizer SGD
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NESTEROV = False

# Early stopping
EARLY_STOPPING_PATIENCE = 20
EARLY_STOPPING_MIN_DELTA = 0.0

# DataLoader
NUM_WORKERS = 2
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
PIN_MEMORY = True

# Optional testing after training
RUN_TEST_AFTER_TRAINING = False
MAX_TEST_SAMPLES = None  # Use 8 for a quick smoke test.
TEST_BATCH_SIZE = 2
TEST_NUM_WORKERS = 0

print(f"Dataset: {DATASET_ROOT}")
print(f"Output: {DRIVE_OUTPUT_DIR}")

## 4. Clone or update the repository

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    print("Updating the existing repository...")
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "pull", "--ff-only",
            "origin", REPOSITORY_BRANCH,
        ],
        check=True,
    )
else:
    print("Cloning the repository...")
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

training_script = PROJECT_ROOT / "003_classification/cv_resnet50/train.py"
if not training_script.is_file():
    raise FileNotFoundError(f"Training script not found: {training_script}")

print(f"Repository is ready: {PROJECT_ROOT}")

## 5. Install the required packages

The PyTorch installation provided by Colab is kept because it already matches the CUDA version in the active runtime.

In [ ]:
import sys

packages = [
    "albumentations>=2.0,<3.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "zennit==0.5.1",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("All dependencies are ready.")

## 6. Copy and extract the dataset

The expected archive is `a0d90f9e-3dd4-4de0-98af-12858696f613_cv_resnet50_parenchyma.tar.gz`, with `000_dataset_v2` as its top-level directory. Copying and extraction display `tqdm` progress bars. Set `FORCE_EXTRACT=True` only when the local extracted data must be recreated.

In [ ]:
import shutil
import tarfile

from tqdm.auto import tqdm


def copy_file_with_progress(source, destination, chunk_size=8 * 1024 * 1024):
    """Copy one file while displaying byte-level progress."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".part")
    total_bytes = source.stat().st_size

    with source.open("rb") as input_file:
        with temporary_path.open("wb") as output_file:
            with tqdm(
                total=total_bytes,
                desc="Copying archive",
                unit="B",
                unit_scale=True,
            ) as progress_bar:
                while True:
                    chunk = input_file.read(chunk_size)
                    if not chunk:
                        break
                    output_file.write(chunk)
                    progress_bar.update(len(chunk))

    temporary_path.replace(destination)


if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DRIVE_ARCHIVE_PATH}")

if COPY_ARCHIVE_TO_LOCAL:
    archive_path = LOCAL_ARCHIVE_PATH
    source_size = DRIVE_ARCHIVE_PATH.stat().st_size
    local_copy_is_current = (
        archive_path.is_file() and archive_path.stat().st_size == source_size
    )

    if local_copy_is_current:
        print(f"Using local archive: {archive_path}")
    else:
        copy_file_with_progress(DRIVE_ARCHIVE_PATH, archive_path)
else:
    archive_path = DRIVE_ARCHIVE_PATH

extraction_marker = EXTRACTION_ROOT / ".extraction_complete"

if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

if extraction_marker.is_file():
    print(f"Using local dataset: {EXTRACTION_ROOT}")
else:
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive_path, mode="r:gz") as archive:
        members = archive.getmembers()
        for member in tqdm(members, desc="Extracting dataset", unit="file"):
            archive.extract(member, path=EXTRACTION_ROOT, filter="data")

    extraction_marker.touch()

print(f"Dataset is ready: {DATASET_ROOT}")

## 7. Validate the `000_dataset_v2` dataset

This cell checks the CV metadata columns, five-fold assignment, classes, and all parenchyma files used for training. Masks are also checked because they are required when the optional XAI test is run.

In [ ]:
import csv

metadata_path = DATASET_ROOT / METADATA_FILENAME
required_directories = [
    EXTRACTION_ROOT / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask",
    EXTRACTION_ROOT / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask",
]
required_columns = {
    "dataset", "patient_id", "filename", CT_PATH_COLUMN,
    "mask_path", "label", "cv_group_id", "cv_nodule_id",
    "cv_role", "cv_fold",
}

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")

for directory in required_directories:
    if not directory.is_dir():
        raise FileNotFoundError(f"Directory not found: {directory}")

with metadata_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    column_names = set(reader.fieldnames or [])
    rows = list(reader)

missing_columns = required_columns - column_names
if missing_columns:
    raise ValueError(f"Metadata columns are incomplete: {sorted(missing_columns)}")
if not rows:
    raise ValueError("Metadata must not be empty.")

missing_files = []
for row in tqdm(rows, desc="Checking dataset paths", unit="sample"):
    ct_path = DATASET_ROOT / row[CT_PATH_COLUMN]
    mask_path = DATASET_ROOT / row["mask_path"]

    if not ct_path.is_file():
        missing_files.append(ct_path)
    if not mask_path.is_file():
        missing_files.append(mask_path)

if missing_files:
    examples = "\n".join(str(path) for path in missing_files[:5])
    raise FileNotFoundError(
        f"There are {len(missing_files)} missing files.\n{examples}"
    )

development_folds = {
    int(row["cv_fold"])
    for row in rows
    if row["cv_role"].strip().lower() == "development"
}
if development_folds != set(range(NUM_FOLDS)):
    raise ValueError(f"Unexpected folds: {sorted(development_folds)}")

label_counts = {}
for row in rows:
    label = row["label"].strip().lower()
    label_counts[label] = label_counts.get(label, 0) + 1

print(f"Number of samples: {len(rows):,}")
print(f"Class distribution: {label_counts}")
print(f"Development folds: {sorted(development_folds)}")
print("All parenchyma and mask files are available.")

## 8. Create and save the JSON configuration

The dictionary below is created directly from the main configuration cell. The JSON file is saved to the temporary repository so `train.py` can read it, and a persistent copy is saved to Google Drive. When training starts, `train.py` also copies the same JSON file to the experiment output directory as `cv_resnet50.json`.

In [ ]:
import json

if NUM_FOLDS != 5:
    raise ValueError("This metadata requires NUM_FOLDS = 5.")
if BATCH_SIZE < 1 or NUM_EPOCHS < 1 or LEARNING_RATE <= 0:
    raise ValueError("Batch size, epochs, and learning rate must be positive.")
if NUM_WORKERS < 0:
    raise ValueError("NUM_WORKERS must not be negative.")
if not 0.0 <= CLASSIFICATION_THRESHOLD <= 1.0:
    raise ValueError("CLASSIFICATION_THRESHOLD must be between 0 and 1.")
if DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("DEVICE must be 'auto', 'cpu', or 'cuda'.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda', but a GPU is not available.")
config = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "component": EXPERIMENT_COMPONENT,
    },
    "output": {
        "root_directory": str(DRIVE_OUTPUT_ROOT),
        "config_snapshot_filename": "cv_resnet50.json",
    },
    "data": {
        "dataset_root": str(DATASET_ROOT),
        "metadata_path": str(metadata_path),
        "ct_path_column": CT_PATH_COLUMN,
        "input_height": INPUT_HEIGHT,
        "input_width": INPUT_WIDTH,
        "class_to_idx": CLASS_TO_IDX,
        "normalization_mean": NORMALIZATION_MEAN,
        "normalization_std": NORMALIZATION_STD,
    },
    "cross_validation": {
        "num_folds": NUM_FOLDS,
        "development_role": "development",
        "holdout_role": "holdout_test",
        "holdout_fold": -1,
        "group_column": "cv_group_id",
        "nodule_column": "cv_nodule_id",
        "fold_column": "cv_fold",
        "role_column": "cv_role",
    },
    "model": {
        "architecture": "ResNet50",
        "pretrained_weights": PRETRAINED_WEIGHTS,
        "training_strategy": "full_fine_tuning",
        "trainable_component": "entire_model",
        "classifier_dropout": CLASSIFIER_DROPOUT,
    },
    "training": {
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "transform_seed": TRANSFORM_SEED,
        "classification_threshold": CLASSIFICATION_THRESHOLD,
        "device": DEVICE,
    },
    "optimizer": {
        "name": "SGD",
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "nesterov": NESTEROV,
    },
    "dataloader": {
        "num_workers": NUM_WORKERS,
        "persistent_workers": PERSISTENT_WORKERS,
        "prefetch_factor": PREFETCH_FACTOR,
        "pin_memory": PIN_MEMORY,
        "train_shuffle": True,
        "val_shuffle": False,
        "train_drop_last": False,
        "val_drop_last": False,
    },
    "early_stopping": {
        "enabled": True,
        "monitor": "val_loss",
        "mode": "min",
        "patience": EARLY_STOPPING_PATIENCE,
        "min_delta": EARLY_STOPPING_MIN_DELTA,
        "verbose": True,
        "restore_best_weights": True,
    },
    "checkpoint": {"save_latest": True},
}


def save_json(data, output_path):
    """Save a dictionary as a readable JSON file."""

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)
        file.write("\n")


config_path = PROJECT_ROOT / "003_classification/configs/cv_resnet50.json"
drive_config_path = (
    DRIVE_OUTPUT_ROOT / "saved_configs" / EXPERIMENT_ID / "cv_resnet50.json"
)
save_json(config, config_path)
save_json(config, drive_config_path)

print(json.dumps(config, indent=4))
print(f"Config repository: {config_path}")
print(f"Config Google Drive: {drive_config_path}")

## 9. Run the preflight check

The preflight check uses the same configuration and transforms as training. This cell loads one validation sample, builds ResNet-50, and verifies the model input and output shapes. The pretrained ImageNet weights are downloaded when the model is first created.

In [ ]:
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

importlib.invalidate_caches()
module_name = "003_classification.cv_resnet50.train"
if module_name in sys.modules:
    train_module = importlib.reload(sys.modules[module_name])
else:
    train_module = importlib.import_module(module_name)

train_loader, val_loader, _, _ = (
    train_module.build_fold_dataloaders(fold=0)
)
sample_image, sample_label = val_loader.dataset[0]
model = train_module.build_model(
    num_classes=len(CLASS_TO_IDX)
)
model.eval()

with torch.no_grad():
    sample_output = model(
        sample_image.unsqueeze(0).to(train_module.DEVICE)
    )

print(f"Fold 0 training samples: {len(train_loader.dataset):,}")
print(f"Fold 0 validation samples: {len(val_loader.dataset):,}")
print(f"Input shape: {tuple(sample_image.shape)}")
print(f"Output shape: {tuple(sample_output.shape)}")
print(f"Sample label: {int(sample_label)}")

del model
del train_loader
del val_loader
torch.cuda.empty_cache()

## 10. Start 5-fold training

Training runs in the notebook kernel, so the `tqdm` progress bars for training and validation appear directly below this cell. Each fold uses a new model. The best model is selected by validation loss, while the latest checkpoint is saved after every epoch.

In [ ]:
if DRIVE_OUTPUT_DIR.exists():
    raise FileExistsError(
        f"Output already exists: {DRIVE_OUTPUT_DIR}\n"
        "Change EXPERIMENT_ID to start a new training run."
    )

print("Starting baseline ResNet-50...", flush=True)
print(f"Input: {CT_PATH_COLUMN}", flush=True)
print(f"Fold: {NUM_FOLDS}", flush=True)
print(f"Epochs per fold: {NUM_EPOCHS}", flush=True)
print(f"Output: {DRIVE_OUTPUT_DIR}", flush=True)

train_module.main()

## 11. View the cross-validation results

This cell displays the summary for each fold, the combined plot, and the locations of important artifacts.

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary_path = DRIVE_OUTPUT_DIR / "cv_summary.csv"
plot_path = DRIVE_OUTPUT_DIR / "figures/cv_fold_metrics.png"
config_snapshot_path = DRIVE_OUTPUT_DIR / "cv_resnet50.json"
oof_predictions_path = DRIVE_OUTPUT_DIR / "out_of_fold_predictions.csv"

if not summary_path.is_file():
    raise FileNotFoundError(f"Training summary not found: {summary_path}")

display(pd.read_csv(summary_path))
if plot_path.is_file():
    display(Image(filename=str(plot_path), width=750))

print(f"Output directory: {DRIVE_OUTPUT_DIR}")
print(f"Config snapshot: {config_snapshot_path}")
print(f"OOF predictions: {oof_predictions_path}")
for fold in range(NUM_FOLDS):
    best_model_path = DRIVE_OUTPUT_DIR / f"fold_{fold}/best_model.pth"
    print(f"Fold {fold} best model: {best_model_path}")

## 12. Holdout testing and XAI (optional)

Set `RUN_TEST_AFTER_TRAINING=True` in the configuration cell to evaluate the five-model ensemble on the holdout test set and generate Grad-CAM and LRP results. XAI processing for the complete holdout set may take a long time; use `MAX_TEST_SAMPLES=8` for a smoke test. The test script also displays `tqdm` progress bars.

In [ ]:
if RUN_TEST_AFTER_TRAINING:
    test_command = [
        sys.executable,
        "-m",
        "003_classification.cv_resnet50.test",
        str(DRIVE_OUTPUT_DIR),
        "--batch-size",
        str(TEST_BATCH_SIZE),
        "--num-workers",
        str(TEST_NUM_WORKERS),
        "--device",
        DEVICE,
        "--dpi",
        "120",
    ]

    if MAX_TEST_SAMPLES is not None:
        test_command.extend(["--max-samples", str(MAX_TEST_SAMPLES)])

    subprocess.run(test_command, cwd=PROJECT_ROOT, check=True)
    print(f"Test results: {DRIVE_OUTPUT_DIR / 'test'}")
else:
    print("Testing was skipped because RUN_TEST_AFTER_TRAINING=False.")

## Run a new experiment

The current CV program creates a new output directory and does not automatically resume an interrupted fold. To run a new experiment, change `EXPERIMENT_ID`, adjust the configuration, and rerun the notebook starting from the configuration cell. Do not delete previous results from Google Drive until all required artifacts have been saved.